# Transcript Concordance by Biotype

Stacked horizontal bar plots showing transcript concordance categories
(full, partial, none) broken down by biotype. Three panels:
- **Left**: RBH-pass genes only (coverage ≥ 95%)
- **Middle**: All genes with an RBH pair (pass + fail, regardless of coverage threshold)
- **Right**: All Ensembl genes — RBH-paired genes with their actual concordance **plus**
  Ensembl-only genes (no CAT pair in a given assembly) counted as 'No match'.
  Uses `sankey_level1_gene_presence.tsv` to add the Ensembl-only counts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

# PDF font settings — TrueType so text is editable in Illustrator / Inkscape
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

# Configuration
RESULTS_DIR = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results')
LINKS_DIR  = RESULTS_DIR / 'intermediate_spreadsheets' / 'sankey_plus_divergence'
GP_DIR     = RESULTS_DIR / 'intermediate_spreadsheets' / 'gene_presence'
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

# Colour scheme (consistent with sankey figure)
CONCORDANCE_COLORS = {
    'full':    '#1b4f72',   # Dark blue
    'partial': '#85c1e9',   # Light blue
    'none':    '#e74c3c',   # Red
}
CONCORDANCE_ORDER  = ['full', 'partial', 'none']
CONCORDANCE_LABELS = {'full': 'Full match', 'partial': 'Partial', 'none': 'No match'}

BIOTYPE_ORDER  = ['protein_coding', 'lncRNA', 'pseudogene', 'other_ncRNA', 'other']
BIOTYPE_LABELS = {
    'protein_coding': 'Protein-coding',
    'lncRNA':         'lncRNA',
    'pseudogene':     'Pseudogene',
    'other_ncRNA':    'Other ncRNA',
    'other':          'Other',
}

def group_biotype(b: str) -> str:
    b = str(b or '').lower()
    if 'protein_coding' in b:
        return 'protein_coding'
    if 'lncrna' in b or 'lnc_rna' in b:
        return 'lncRNA'
    if 'pseudogene' in b or 'pseudogenic' in b:
        return 'pseudogene'
    if any(x in b for x in ['snrna', 'snorna', 'mirna', 'trna', 'rrna',
                             'ncrna', 'antisense', 'tec', 'guide_rna',
                             'scrna', 'vault_rna', 'y_rna']):
        return 'other_ncRNA'
    return 'other'

In [ ]:
# ── Load per-gene RBH links ──────────────────────────────────────────────────
links = pd.read_csv(LINKS_DIR / 'links_per_assembly.tsv', sep='\t')
print(f'links_per_assembly.tsv: {len(links):,} rows')
print(f'  Assemblies : {links["assembly_accession"].nunique()}')
print(f'  rbh_status : {dict(links["rbh_status"].value_counts())}')
print(f'  Columns    : {list(links.columns)}')

In [ ]:
# ── Load gene presence (for Ensembl-only genes with no RBH pair) ─────────────
gp = pd.read_csv(GP_DIR / 'sankey_level1_gene_presence.tsv', sep='\t')
print(f'sankey_level1_gene_presence.tsv: {len(gp):,} rows')
print(f'  cohort_status counts: {dict(gp["cohort_status"].value_counts())}')
print(f'  Columns: {list(gp.columns)}')

In [ ]:
# ── Build cross-tabs for each panel ─────────────────────────────────────────

def make_crosstab(df, rbh_statuses):
    """Cross-tab of biotype × tx_concordance for given rbh_statuses."""
    subset = df[df['rbh_status'].isin(rbh_statuses)].dropna(subset=['tx_concordance'])
    return (
        subset
        .groupby(['biotype', 'tx_concordance'])
        .size()
        .unstack(fill_value=0)
        .reindex(index=BIOTYPE_ORDER, columns=CONCORDANCE_ORDER, fill_value=0)
    )

# Panel A – RBH-pass only
ct_pass = make_crosstab(links, ['pass'])
print(f'Panel A (RBH-pass):\n{ct_pass}\n')

# Panel B – All RBH pairs (pass + fail)
ct_all_rbh = make_crosstab(links, ['pass', 'fail'])
print(f'Panel B (all RBH):\n{ct_all_rbh}\n')

# Panel C – All Ensembl genes:
#   start from all-RBH concordance counts, then ADD Ensembl-only
#   genes (no CAT pair in a given assembly) as extra 'none' entries.
#
#   sankey_level1_gene_presence.tsv stores how many assemblies each
#   Ensembl gene was present without a CAT match → n_assemblies_ensembl_only.
#   We expand those into biotype × 'none' contributions.

# Bucket the Ensembl biotype from the presence file
gp['biotype_grouped'] = gp['ensembl_biotype'].map(group_biotype)

# Only consider genes that appear as Ensembl-only in ≥1 assembly
ens_only = gp[gp['n_assemblies_ensembl_only'] > 0].copy()

# Sum weighted none-concordance counts per biotype bucket
extra_none = (
    ens_only
    .groupby('biotype_grouped')['n_assemblies_ensembl_only']
    .sum()
    .reindex(BIOTYPE_ORDER, fill_value=0)
)
print(f'Extra "none" from Ensembl-only genes (per biotype bucket):\n{extra_none}\n')

# Build Panel C by copying Panel B and adding extra 'none' counts
ct_all_genes = ct_all_rbh.copy()
for bio in BIOTYPE_ORDER:
    ct_all_genes.loc[bio, 'none'] += int(extra_none.loc[bio])

print(f'Panel C (all Ensembl genes):\n{ct_all_genes}')

In [ ]:
# ── Quick sanity: total gene-assembly counts ─────────────────────────────────
n_assemblies = links['assembly_accession'].nunique()

print(f'Assemblies: {n_assemblies}')
print(f'Panel A total gene-assembly pairs: {ct_pass.values.sum():,}')
print(f'Panel B total gene-assembly pairs: {ct_all_rbh.values.sum():,}')
print(f'Panel C total gene-assembly pairs: {ct_all_genes.values.sum():,}')
print(f'Extra none added: {extra_none.sum():,}')

In [ ]:
# ── Plotting ─────────────────────────────────────────────────────────────────

def draw_biotype_concordance_bar(ax, ct, title, label_threshold_pct=8):
    """Draw a stacked horizontal bar chart on *ax*."""
    y_labels = [BIOTYPE_LABELS.get(b, b) for b in BIOTYPE_ORDER]
    y_pos    = np.arange(len(BIOTYPE_ORDER))
    left     = np.zeros(len(BIOTYPE_ORDER))

    for conc in CONCORDANCE_ORDER:
        vals = ct[conc].values.astype(float)
        ax.barh(
            y_pos, vals, left=left,
            color=CONCORDANCE_COLORS[conc],
            label=CONCORDANCE_LABELS[conc],
            edgecolor='white', linewidth=0.5,
        )
        for j, (v, l) in enumerate(zip(vals, left)):
            total_row = float(ct.loc[BIOTYPE_ORDER[j]].sum())
            if total_row == 0:
                continue
            pct = v / total_row * 100
            if pct >= label_threshold_pct:
                ax.text(
                    l + v / 2, j,
                    f'{int(v):,}\n({pct:.0f}%)',
                    ha='center', va='center',
                    fontsize=7.5, fontweight='bold', color='white',
                )
        left += vals

    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=10)
    ax.set_xlabel('Gene-assembly count', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8.5, framealpha=0.9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.invert_yaxis()  # protein-coding at top


fig, axes = plt.subplots(1, 3, figsize=(26, 5.5), sharey=True)
ax_pass, ax_rbh, ax_all = axes

draw_biotype_concordance_bar(ax_pass, ct_pass,     'RBH-pass (coverage ≥ 95%)')
draw_biotype_concordance_bar(ax_rbh,  ct_all_rbh,  'All RBH pairs (pass + fail)')
draw_biotype_concordance_bar(ax_all,  ct_all_genes, 'All Ensembl genes\n(RBH-paired + unmatched Ensembl-only)')

# Remove duplicate y-tick labels on shared-axis panels
ax_rbh.set_yticklabels([])
ax_all.set_yticklabels([])

# ── Criteria text ─────────────────────────────────────────────────────────────
criteria = (
    'Criteria \u2014 '
    'Left: reciprocal best-hit (RBH) pairs with locus coverage \u2265 95% on both Ensembl and CAT sides. '
    'Middle: all RBH pairs regardless of coverage. '
    'Right (new): all genes above + Ensembl-only genes (no CAT RBH pair in a given assembly) '
    'counted as \u201cNo match\u201d using assembly-weighted counts from sankey_level1_gene_presence.tsv. '
    'Full match: transcript concordance rate = 1.0 (all exon structures matched bidirectionally). '
    'Partial: 0 < concordance < 1.0. '
    'No match: concordance = 0 OR gene present only in Ensembl for that assembly. '
    'Biotypes: protein_coding; lncRNA (incl. lnc_RNA); pseudogene (incl. pseudogenic); '
    'other_ncRNA (snRNA, snoRNA, miRNA, tRNA, rRNA, etc.); other (all remaining). '
    'Counts summed across all pangenome assemblies.'
)
fig.text(0.5, -0.05, criteria, ha='center', fontsize=7.5, color='#555555',
         style='italic', wrap=True)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_main3_biotype_concordance.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_main3_biotype_concordance.pdf', bbox_inches='tight')
plt.show()
print(f'Saved to {OUTPUT_DIR / "figure_main3_biotype_concordance.png"}')
print(f'Saved to {OUTPUT_DIR / "figure_main3_biotype_concordance.pdf"}')

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
# Useful for checking numbers and for paper supplementary tables.

def pct_col(ct):
    totals = ct.sum(axis=1)
    return ct.div(totals, axis=0).mul(100).round(1)

panels = {
    'RBH-pass': ct_pass,
    'All-RBH':  ct_all_rbh,
    'All-genes': ct_all_genes,
}

rows = []
for panel_name, ct in panels.items():
    pct = pct_col(ct)
    for bio in BIOTYPE_ORDER:
        row = {'panel': panel_name, 'biotype': BIOTYPE_LABELS[bio]}
        total = int(ct.loc[bio].sum())
        row['total'] = total
        for conc in CONCORDANCE_ORDER:
            row[f'n_{conc}'] = int(ct.loc[bio, conc])
            row[f'pct_{conc}'] = float(pct.loc[bio, conc])
        rows.append(row)

summary = pd.DataFrame(rows)
display(summary)